In [1]:
from mpramnist.Ernst2016 import ErnstDataset
from mpramnist.Ernst2016 import LitModel_Ernst

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

from mpramnist import transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1096
NUM_WORKERS = 8

print(ErnstDataset.CELL_TYPES)

['k562_minp_rep1', 'k562_minp_rep2', 'k562_minp_avg', 'k562_sv40p_rep1', 'k562_sv40p_rep2', 'k562_sv40p_avg', 'hepg2_minp_rep1', 'hepg2_minp_rep2', 'hepg2_minp_avg', 'hepg2_sv40p_rep1', 'hepg2_sv40p_rep2', 'hepg2_sv40p_avg']


/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# preprocessing
train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
test_transform = t.Compose([t.Seq2Tensor(),])
# load the data
cell_type = [
    "k562_minp_avg",
    "k562_sv40p_avg",
    "hepg2_minp_avg",
    "hepg2_sv40p_avg",
]
train_dataset = ErnstDataset(split="train",cell_type=cell_type,transform=train_transform,root="../data/",)  # for needed folds
val_dataset = ErnstDataset(split="val",cell_type=cell_type,transform=test_transform,root="../data/",)  # use "val" for default validation set
test_dataset = ErnstDataset(split="test",cell_type=cell_type,transform=test_transform,root="../data/",)  # use "test" for default test set

print(len(train_dataset),len(val_dataset),len(test_dataset))

# encapsulate data into dataloader form
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = len(cell_type)

val: After filtering duplicates: 19833 sequences in val
test: After filtering duplicates: 10130 sequences in test
457174 19833 10130


In [3]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Ernst(model=model,loss=nn.MSELoss(),cell_types=cell_type,weight_decay=1e-1,lr=1e-2,print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=False,
    num_sanity_val_steps=0,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | 


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 0.93105 
| Val Pearson k562_minp_avg: 0.36537 | Val Pearson k562_sv40p_avg: 0.15930 | Val Pearson hepg2_minp_avg: 0.25612 | Val Pearson hepg2_sv40p_avg: 0.20801 | Mean Val Pearson: 0.24720 |
| Train Pearson k562_minp_avg: 0.35210 | Train Pearson k562_sv40p_avg: 0.16611 | Train Pearson hepg2_minp_avg: 0.24487 | Train Pearson hepg2_sv40p_avg: 0.20352 | Mean Train Pearson: 0.24165 |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------



In [4]:
def meaned_prediction(forw, rev, trainer, seq_model, name, num):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef(num_outputs=num)
    print(name + " Pearson correlation")

    return pears(mean_forw, targets)

forw_transform = t.Compose([t.Seq2Tensor()])
rev_transform = t.Compose([t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = ErnstDataset(split="test",cell_type=cell_type,transform=forw_transform,root="../data/",)
test_rev = ErnstDataset(split="test",cell_type=cell_type,transform=rev_transform,root="../data/",)

forw = DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw, rev, trainer, seq_model, "Sharpr", len(cell_type))

test: After filtering duplicates: 10130 sequences in test


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


test: After filtering duplicates: 10130 sequences in test


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Sharpr Pearson correlation


tensor([0.3549, 0.1830, 0.2407, 0.2591])